# 🧠 Meeting Intelligence NLP Model
### Fr. C. Rodrigues Institute of Technology, Vashi, Navi-Mumbai
**Course:** CELBC608 – Data Science Laboratory  
**Academic Year:** 2025-26 | **Semester:** VI  
**Faculty:** Mrs. Smita R., Mrs. Nupur G.

---
**Project Goal:** Automatically classify meeting transcripts into categories: `decision`, `action`, `issue`, `deadline`, or `general` using NLP + Machine Learning.

**Experiments Covered:** Exp 1 → Exp 7 (as per lab syllabus)

| Exp | Topic | CO Mapped |
|-----|-------|-----------|
| 1 | Data Loading & Exploration | CO-1 |
| 2 | Data Preprocessing | CO-2 |
| 3 | Feature Extraction & Train-Test Split | CO-3 |
| 4 | Model Training (LinearSVC) | CO-3 |
| 5 | Evaluation – Accuracy, Confusion Matrix, Classification Report | CO-3 |
| 6 | NLP Labeling (Rule-Based) | CO-4 |
| 7 | Text Analysis – Length Distribution | CO-4 |

## ⚙️ Step 0: Install Dependencies
Install all required libraries before running the experiments.

In [ ]:
!pip install datasets scikit-learn pandas matplotlib seaborn nltk PyPDF2 python-docx streamlit pyngrok -q

---
## 🔸 Exp 1: Data Loading & Initial Exploration
**CO Mapped:** CO-1 | **Cognitive Level:** Apply

### What?
Load the DialogSum dataset from HuggingFace and explore its structure.

### Why?
Before any ML pipeline, we must understand our data — its shape, missing values, data types, and sample content.

### Expected Output:
- Dataset shape
- Sample rows
- Null value check

In [ ]:
# ============================================================
# EXP 1 — DATA LOADING & INITIAL EXPLORATION
# ============================================================

from datasets import load_dataset
import pandas as pd

# Load DialogSum dataset from HuggingFace
dataset = load_dataset("knkarthick/dialogsum")

# Convert training split to DataFrame
df = pd.DataFrame(dataset['train'])

# Keep only the 'dialogue' column and rename to 'text'
df = df[['dialogue']].rename(columns={'dialogue': 'text'})

# Save for later use
df.to_csv("dialogsum_data.csv", index=False)

print("✅ Dataset Loaded Successfully!")
print(f"Shape: {df.shape}")
print(f"\nMissing Values:\n{df.isnull().sum()}")
print("\n--- Sample Data (First 5 Rows) ---")
df.head()

---
## 🔸 Exp 6: NLP Labeling (Rule-Based)
**CO Mapped:** CO-4 | **Cognitive Level:** Analyze

### What?
Assign a category label to each dialogue using a rule-based keyword matching approach.

### Why?
Since the dataset has no ground-truth labels for meeting categories, we create them programmatically using domain knowledge. This is a common technique in real-world NLP pipelines.

### Rules:
| Label | Trigger Keywords |
|-------|------------------|
| `decision` | decide, decided, agreed, approved, confirmed, finalized |
| `action` | will, task, implement, assign, prepare, handle, execute |
| `issue` | issue, problem, error, bug, risk, failure, challenge |
| `deadline` | deadline, by, due, before, tomorrow, asap, next week |
| `general` | (default fallback) |

In [ ]:
# ============================================================
# EXP 6 — NLP LABELING (Rule-Based)
# ============================================================

def label_text(text):
    """
    Rule-based function to assign a meeting category label.
    Priority order: decision > action > issue > deadline > general
    """
    text = text.lower()

    decision_words = [
        "decide", "decided", "decision", "agreed", "finalized",
        "approved", "confirmed", "concluded", "let's", "we should"
    ]
    action_words = [
        "will", "task", "action", "complete", "finish",
        "implement", "execute", "prepare", "assign", "handle"
    ]
    issue_words = [
        "issue", "problem", "error", "bug", "risk",
        "challenge", "conflict", "failure", "wrong"
    ]
    deadline_words = [
        "deadline", "by", "due", "before", "within",
        "tomorrow", "next week", "today", "asap"
    ]

    if any(w in text for w in decision_words):
        return "decision"
    elif any(w in text for w in action_words):
        return "action"
    elif any(w in text for w in issue_words):
        return "issue"
    elif any(w in text for w in deadline_words):
        return "deadline"
    else:
        return "general"

# Apply labeling to entire dataframe
df['label'] = df['text'].apply(label_text)

print("✅ Labels assigned!")
print("\n--- Label Distribution ---")
print(df['label'].value_counts())

---
## 🔸 Exp 2: Data Preprocessing
**CO Mapped:** CO-2 | **Cognitive Level:** Apply

### What?
Clean the raw text data to remove noise and standardize it for ML training.

### Why?
Raw text contains special characters, numbers, and mixed cases that confuse ML models. Cleaning improves model accuracy.

### Steps:
1. Convert text to lowercase
2. Remove all non-alphabetic characters using regex
3. Filter out very short texts (less than 10 characters)
4. Remove `general` label rows (sparse class, affects model quality)

In [ ]:
# ============================================================
# EXP 2 — DATA PREPROCESSING
# ============================================================

import re

def clean_text(text):
    """
    Preprocessing function:
    - Lowercases text
    - Removes non-alphabetic characters
    Returns: cleaned string
    """
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # remove numbers, punctuation
    return text

# Apply cleaning
df['clean_text'] = df['text'].apply(clean_text)

# Filter out very short texts
df = df[df['clean_text'].str.len() > 10]

# Remove 'general' category (too vague, reduces model quality)
df = df[df['label'] != 'general']

print("✅ Preprocessing Complete!")
print(f"Remaining samples after filtering: {len(df)}")
print("\n--- Before vs After Cleaning (First 3 rows) ---")
df[['text', 'clean_text', 'label']].head(3)

---
## 🔸 Exp 7: Text Analysis
**CO Mapped:** CO-4 | **Cognitive Level:** Analyze

### What?
Analyze the distribution of text lengths in the cleaned dataset and visualize label distribution.

### Why?
Text length distribution helps us understand dataset characteristics — very short texts may carry insufficient signal for classification, while extremely long texts may dominate TF-IDF features.

In [ ]:
# ============================================================
# EXP 7 — TEXT ANALYSIS
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns

# ---- Plot 1: Text Length Distribution ----
df['length'] = df['clean_text'].apply(len)

plt.figure(figsize=(10, 4))
sns.histplot(df['length'], bins=40, kde=True, color='steelblue')
plt.title("Text Length Distribution (After Cleaning)")
plt.xlabel("Text Length (characters)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

# ---- Plot 2: Label Distribution ----
plt.figure(figsize=(8, 4))
df['label'].value_counts().plot(kind='bar', color=['#4C72B0','#DD8452','#55A868','#C44E52'], edgecolor='black')
plt.title("Label Distribution Across Meeting Categories")
plt.xlabel("Category")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print("\nLabel Counts:")
print(df['label'].value_counts())
print(f"\nAverage text length: {df['length'].mean():.0f} characters")

---
## 🔸 Exp 3: Feature Extraction & Train-Test Split
**CO Mapped:** CO-3 | **Cognitive Level:** Analyze

### What?
Convert text into numerical features using TF-IDF Vectorization, then split into training and testing sets.

### Why?
Machine learning models cannot directly process raw text — they require numbers. TF-IDF represents each word by its importance in a document relative to the corpus.

### TF-IDF Parameters Used:
| Parameter | Value | Reason |
|-----------|-------|--------|
| `max_features` | 8000 | Keeps top 8000 informative words |
| `ngram_range` | (1,2) | Uses single words + bigrams for better context |
| `stop_words` | 'english' | Removes common filler words |
| `min_df` | 3 | Ignores very rare words (noise reduction) |
| `max_df` | 0.85 | Ignores words appearing in >85% of docs |

In [ ]:
# ============================================================
# EXP 3 — FEATURE EXTRACTION (TF-IDF)
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# TF-IDF Vectorizer with improved parameters
vectorizer = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 2),    # 🔥 unigrams + bigrams
    stop_words='english',
    min_df=3,              # remove rare tokens
    max_df=0.85            # remove overly common tokens
)

# Feature matrix
X = vectorizer.fit_transform(df['clean_text'])

# Target labels
y = df['label']

print("✅ TF-IDF Vectorization Complete!")
print(f"Feature Matrix Shape (X): {X.shape}")
print(f"Target Vector Shape (y): {y.shape}")
print(f"\nSample features: {vectorizer.get_feature_names_out()[:10]}")

# ---- Train-Test Split ----
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y     # ensures balanced class distribution in both splits
)

print(f"\nTraining set shape: {X_train.shape}")
print(f"Testing set shape:  {X_test.shape}")
print(f"\nLabel distribution in training set:")
print(y_train.value_counts())

---
## 🔸 Exp 4: Model Training
**CO Mapped:** CO-3 | **Cognitive Level:** Apply

### What?
Train a **LinearSVC** (Support Vector Classifier) on the TF-IDF features.

### Why LinearSVC?
- **Fast** on large sparse text datasets
- **High accuracy** for multi-class text classification
- **Robust** to class imbalance when tuned with `C` parameter
- Outperforms Logistic Regression and Naive Bayes on NLP tasks

### Hyperparameter:
- `C=1.2` — Regularization strength. Higher = less regularization, better fit on clean data.

In [ ]:
# ============================================================
# EXP 4 — MODEL TRAINING (LinearSVC)
# ============================================================

from sklearn.svm import LinearSVC

# Initialize model
model = LinearSVC(C=1.2)

# Train on training data
model.fit(X_train, y_train)

print("✅ LinearSVC Model Trained Successfully!")
print(f"Model type: {type(model).__name__}")
print(f"Trained on {X_train.shape[0]} samples")
print(f"Number of classes: {len(model.classes_)}")
print(f"Classes: {model.classes_}")

---
## 🔸 Exp 5: Model Evaluation
**CO Mapped:** CO-3 | **Cognitive Level:** Analyze

### What?
Evaluate the trained model using multiple metrics:
1. **Accuracy** — overall correct predictions
2. **Confusion Matrix** — per-class prediction breakdown
3. **Classification Report** — precision, recall, F1-score per class

### Why?
Accuracy alone can be misleading in multi-class problems. Precision/Recall/F1 give a per-class view of model quality.

In [ ]:
# ============================================================
# EXP 5 — MODEL EVALUATION
# ============================================================

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# --- Predictions ---
y_pred = model.predict(X_test)

# ---- 1. Accuracy ----
acc = accuracy_score(y_test, y_pred)
print("=" * 50)
print(f"✅ Overall Accuracy: {acc:.4f} ({acc*100:.2f}%)")
print("=" * 50)

# ---- 2. Classification Report ----
print("\n📋 Classification Report:")
print("-" * 50)
print(classification_report(y_test, y_pred, zero_division=0))

# ---- 3. Confusion Matrix ----
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=model.classes_,
    yticklabels=model.classes_
)
plt.title("Confusion Matrix – Meeting Category Classifier")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

---
## 🔸 Sample Predictions

### What?
Run the model on a few hand-crafted sentences to verify it works correctly.

### Why?
Sample predictions let us manually inspect whether the model is learning meaningful patterns — an important step in viva demonstrations.

In [ ]:
# ============================================================
# SAMPLE PREDICTIONS
# ============================================================

sample_texts = [
    "We decided to move the deadline to next Monday after the team agreed.",
    "John will prepare the project report and submit it by Friday.",
    "There is a critical bug in the login module that needs immediate fixing.",
    "The submission deadline is tomorrow morning, everyone must finish asap.",
    "We discussed various points about the upcoming product roadmap."
]

print("🔍 Sample Predictions:")
print("=" * 70)
for i, text in enumerate(sample_texts):
    cleaned = clean_text(text)
    vec = vectorizer.transform([cleaned])
    pred = model.predict(vec)[0]
    print(f"Text {i+1}: {text[:60]}...")
    print(f"  → Predicted Label: [{pred.upper()}]")
    print()

---
## 🔸 Prediction Function (FRONTEND INTEGRATION)

### What?
Define a reusable `predict_text()` function that encapsulates the entire prediction pipeline.

### Why?
The Streamlit frontend app calls this exact function when the user clicks "Analyze". Keeping it clean and reusable is essential for frontend-backend integration.

In [ ]:
# ============================================================
# PREDICTION FUNCTION — Used by Frontend
# ============================================================

def predict_text(text):
    """
    Full prediction pipeline:
    1. Clean the input text
    2. Vectorize using TF-IDF
    3. Predict using trained LinearSVC model

    Args:
        text (str): Raw meeting transcript or sentence

    Returns:
        str: Predicted category label
    """
    cleaned = clean_text(text)
    vectorized = vectorizer.transform([cleaned])
    label = model.predict(vectorized)[0]
    return label


# Quick test
test_input = "We agreed that the team will finalize the report by next week."
print(f"Input: {test_input}")
print(f"Predicted Label: {predict_text(test_input)}")

---
## 🔸 Model Saving

### What?
Serialize the trained model and vectorizer using Python's `pickle` module.

### Why?
The Streamlit frontend app loads `model.pkl` and `vectorizer.pkl` to make predictions without retraining.

### File Paths (DO NOT CHANGE — Frontend depends on these):
- `model.pkl` — LinearSVC trained model
- `vectorizer.pkl` — TF-IDF vectorizer fitted on training data
- `accuracy.pkl` — accuracy score (displayed in frontend UI)

In [ ]:
# ============================================================
# MODEL SAVING
# ============================================================

import pickle
import os

# Save model
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

# Save vectorizer
with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

# Save accuracy (used by frontend)
with open("accuracy.pkl", "wb") as f:
    pickle.dump(acc, f)

print("✅ All files saved successfully!")
print(f"  model.pkl     → {os.path.getsize('model.pkl')/1024:.1f} KB")
print(f"  vectorizer.pkl → {os.path.getsize('vectorizer.pkl')/1024:.1f} KB")
print(f"  accuracy.pkl  → Accuracy = {acc*100:.2f}%")

---
## 🔸 Download Saved Files

Download the trained model files to your local machine for deployment with the Streamlit app.

In [ ]:
from google.colab import files
files.download("model.pkl")
files.download("vectorizer.pkl")
files.download("accuracy.pkl")
files.download("dialogsum_data.csv")

---
## 🔸 Frontend App (UNCHANGED — Original Streamlit UI)

**⚠️ This section is IDENTICAL to the original notebook's frontend.**

The app loads `model.pkl`, `vectorizer.pkl`, `accuracy.pkl`, and `dialogsum_data.csv`.

### Install dependencies for Streamlit deployment:

In [ ]:
!pip install streamlit pyngrok nltk scikit-learn pandas PyPDF2 python-docx -q

### Write the Streamlit App:

In [ ]:
%%writefile app.py

import streamlit as st
import pickle
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize

# Download NLTK
nltk.download('stopwords')
nltk.download('punkt')

# ------------------------
# LOAD MODEL
# ------------------------
with open("model.pkl", "rb") as f:
    model = pickle.load(f)

with open("vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)

with open("accuracy.pkl", "rb") as f:
    accuracy = pickle.load(f)

# ------------------------
# LOAD DATASET
# ------------------------
df = pd.read_csv("dialogsum_data.csv")

stop_words = set(stopwords.words('english'))

# ------------------------
# CLEAN TEXT
# ------------------------
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    return " ".join(tokens)

# ------------------------
# SUMMARY FUNCTION
# ------------------------
def generate_summary(text):
    sentences = sent_tokenize(text)
    
    keywords = ["decide", "will", "important", "issue", "plan", "action"]
    
    important = []
    for s in sentences:
        for k in keywords:
            if k in s.lower():
                important.append(s.strip())
                break

    if len(important) == 0:
        important = sentences[:3]

    return " ".join(important[:5])

# ------------------------
# INSIGHT EXTRACTION
# ------------------------
def extract_insights(text):
    insights = {
        "Decisions": [],
        "Actions": [],
        "Deadlines": [],
        "Issues": []
    }

    sentences = sent_tokenize(text)

    for s in sentences:
        s_clean = s.lower()

        if any(word in s_clean for word in ["decide", "agreed", "final", "approved"]):
            insights["Decisions"].append(s.strip())

        if any(word in s_clean for word in ["will", "need to", "assign", "task", "follow up"]):
            insights["Actions"].append(s.strip())

        if any(word in s_clean for word in ["by", "before", "deadline", "due", "tomorrow", "next"]):
            insights["Deadlines"].append(s.strip())

        if any(word in s_clean for word in ["issue", "problem", "error", "bug", "risk"]):
            insights["Issues"].append(s.strip())

    return insights

# ------------------------
# UI
# ------------------------
st.set_page_config(layout="wide")

st.title("🧠 MeetingMind AI – Smart Meeting Analyzer")

option = st.radio("Choose Input Method", ["Upload File", "Use Dataset"])

text = ""

# ------------------------
# FILE UPLOAD (UPDATED)
# ------------------------
if option == "Upload File":
    uploaded_file = st.file_uploader(
        "Upload file",
        type=["txt", "csv", "pdf", "docx"]
    )

    if uploaded_file is not None:

        # -------- TXT --------
        if uploaded_file.name.endswith(".txt"):
            text = uploaded_file.read().decode("utf-8")

        # -------- CSV --------
        elif uploaded_file.name.endswith(".csv"):
            df_file = pd.read_csv(uploaded_file)
            st.write("Preview:", df_file.head())

            col = st.selectbox("Select text column", df_file.columns)
            text = " ".join(df_file[col].astype(str))

        # -------- PDF --------
        elif uploaded_file.name.endswith(".pdf"):
            from PyPDF2 import PdfReader

            pdf = PdfReader(uploaded_file)
            text = ""

            for page in pdf.pages:
                text += page.extract_text() + " "

        # -------- DOCX --------
        elif uploaded_file.name.endswith(".docx"):
            import docx

            doc = docx.Document(uploaded_file)
            text = ""

            for para in doc.paragraphs:
                text += para.text + " "

        st.text_area("Extracted Text", text, height=200)

# ------------------------
# DATASET OPTION
# ------------------------
elif option == "Use Dataset":
    st.write("Dataset Preview")
    st.dataframe(df.head())

    index = st.slider("Select Meeting", 0, len(df)-1, 0)
    text = df.iloc[index]['text']

    st.text_area("Selected Transcript", text, height=150)

# ------------------------
# ANALYZE BUTTON
# ------------------------
if st.button("🚀 Analyze"):

    if text.strip() == "":
        st.warning("No text found")
    else:
        clean = clean_text(text)
        vec = vectorizer.transform([clean])
        pred = model.predict(vec)[0]

        insights = extract_insights(text)
        summary = generate_summary(text)

        # ------------------------
        # OUTPUT
        # ------------------------
        col1, col2 = st.columns(2)

        with col1:
            st.subheader("📌 Prediction")
            st.success(pred)

            

            st.subheader("📊 Model Performance")
            st.write(f"Accuracy: {round(accuracy*100,2)}%")

            st.subheader("📝 Meeting Summary")
            st.info(summary)

        with col2:
            st.subheader("📈 Insight Distribution")

            counts = {k: len(v) for k, v in insights.items()}
            df_chart = pd.DataFrame(counts.items(), columns=["Type", "Count"])
            st.bar_chart(df_chart.set_index("Type"))


        # ------------------------
        # INSIGHTS
        # ------------------------
        st.subheader("💡 Extracted Insights")

        for key, val in insights.items():
            st.write(f"### {key}")

            if len(val) == 0:
                st.write("No data found")
            else:
                for item in val:
                    st.success(item)

### Set ngrok Auth Token:

In [ ]:
from pyngrok import ngrok

# ⚠️ Replace with your own ngrok auth token from https://dashboard.ngrok.com
ngrok.set_auth_token("YOUR_NGROK_TOKEN_HERE")

### Launch the App:

In [ ]:
from pyngrok import ngrok
import subprocess
import time

# Start streamlit
process = subprocess.Popen(["streamlit", "run", "app.py"])

# Wait for server
time.sleep(5)

# Create public URL
public_url = ngrok.connect(8501)
print("🚀 Your App is Live Here:", public_url)

---
## 📋 Summary of All Experiments

| Exp | Task | Algorithm/Tool | CO | Cognitive Level |
|-----|------|----------------|----|-----------------|
| 1 | Data Loading & Exploration | HuggingFace datasets, Pandas | CO-1 | Apply |
| 2 | Data Preprocessing | regex, string ops | CO-2 | Apply |
| 3 | Feature Extraction & Split | TF-IDF, train_test_split | CO-3 | Analyze |
| 4 | Model Training | LinearSVC | CO-3 | Apply |
| 5 | Evaluation | accuracy, confusion matrix, classification report | CO-3 | Analyze |
| 6 | NLP Labeling | Rule-based keyword matching | CO-4 | Apply |
| 7 | Text Analysis | matplotlib, seaborn | CO-4 | Analyze |

---

## 🎯 Viva Preparation – Key Concepts

**Q1: What is TF-IDF?**  
→ TF-IDF (Term Frequency–Inverse Document Frequency) is a numerical statistic that reflects how important a word is to a document relative to the entire corpus. TF measures word frequency in a document; IDF penalizes words that appear in many documents.

**Q2: Why LinearSVC over Logistic Regression?**  
→ LinearSVC is faster and more memory-efficient on sparse high-dimensional TF-IDF data. It uses a hinge loss function and finds an optimal hyperplane, making it well-suited for text classification.

**Q3: What does stratify=y do in train_test_split?**  
→ It ensures that the proportion of each class label is the same in both training and test sets, preventing biased splits.

**Q4: What is the Confusion Matrix?**  
→ A matrix where rows represent true labels and columns represent predicted labels. The diagonal shows correct predictions; off-diagonal cells show misclassifications.

**Q5: What is Precision vs Recall?**  
→ Precision = TP / (TP + FP) — of all predicted positives, how many are correct.  
→ Recall = TP / (TP + FN) — of all actual positives, how many were found.